Imports and Setup

In [1]:
!pip install pandas numpy scikit-learn joblib psutil
!pip install --upgrade xgboost

import pandas as pd
import numpy as np
import glob, os, zipfile, gc, joblib
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

Dataset Upload and Extraction

In [2]:
print("Upload cabspotting.zip")
uploaded = files.upload()
zip_filename = next(iter(uploaded))
with zipfile.ZipFile(zip_filename, 'r') as zf:
    zf.extractall('cabspotting_data')

Upload cabspotting.zip


Saving cabspottingdata.zip to cabspottingdata.zip


Preprocessing

In [3]:
trace_files = sorted(glob.glob('cabspotting_data/**/*.txt', recursive=True))
trace_files = [f for f in trace_files if '_cabs.txt' not in f]

all_trips = []
for fpath in trace_files:
    cab_id = os.path.basename(fpath).replace('new_', '').replace('.txt', '')
    records = []
    with open(fpath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            if len(parts) != 4: continue
            lat, lon, occ, ts = parts
            records.append((float(lat), float(lon), int(occ), int(ts)))
    if not records: continue

    cab_df = pd.DataFrame(records, columns=['latitude','longitude','occupancy','timestamp'])
    cab_df['timestamp'] = pd.to_datetime(cab_df['timestamp'], unit='s')
    cab_df.sort_values('timestamp', inplace=True)

    cab_df['occ_prev'] = cab_df['occupancy'].shift(1)
    pickups = cab_df[(cab_df['occupancy']==1) & (cab_df['occ_prev']==0)].copy()
    dropoffs = cab_df[(cab_df['occupancy']==0) & (cab_df['occ_prev']==1)].copy()
    if pickups.empty or dropoffs.empty: continue

    pickups.rename(columns={'latitude':'pickup_lat','longitude':'pickup_lon','timestamp':'pickup_time'}, inplace=True)
    dropoffs.rename(columns={'latitude':'dropoff_lat','longitude':'dropoff_lon','timestamp':'dropoff_time'}, inplace=True)

    trips = pd.merge_asof(pickups.sort_values('pickup_time'), dropoffs.sort_values('dropoff_time'),
                         left_on='pickup_time', right_on='dropoff_time', direction='forward')
    trips = trips.dropna(subset=['dropoff_lat','dropoff_lon'])

    if not trips.empty:
        all_trips.append(trips[['pickup_lat','pickup_lon','pickup_time','dropoff_lat','dropoff_lon']])
    del cab_df, pickups, dropoffs, trips; gc.collect()

trips = pd.concat(all_trips, ignore_index=True)

San Francisco Bounded Filtering

In [4]:
SF_LAT_MIN, SF_LAT_MAX = 37.70, 37.81
SF_LON_MIN, SF_LON_MAX = -122.52, -122.35

trips = trips[
    (trips['pickup_lat'].between(SF_LAT_MIN, SF_LAT_MAX)) &
    (trips['pickup_lon'].between(SF_LON_MIN, SF_LON_MAX)) &
    (trips['dropoff_lat'].between(SF_LAT_MIN, SF_LAT_MAX)) &
    (trips['dropoff_lon'].between(SF_LON_MIN, SF_LON_MAX))
]

Feature Engineering

In [5]:
trips['pickup_hour'] = trips['pickup_time'].dt.hour.astype(np.int8)
trips['pickup_dayofweek'] = trips['pickup_time'].dt.dayofweek.astype(np.int8)

X = pd.DataFrame(index=trips.index)
X['pickup_lat'] = trips['pickup_lat'].astype(np.float32)
X['pickup_lon'] = trips['pickup_lon'].astype(np.float32)
X['pickup_hour'] = trips['pickup_hour']
X['pickup_dayofweek'] = trips['pickup_dayofweek']

X['hour_sin'] = np.sin(2 * np.pi * X['pickup_hour'] / 24).astype(np.float32)
X['hour_cos'] = np.cos(2 * np.pi * X['pickup_hour'] / 24).astype(np.float32)
X['day_sin'] = np.sin(2 * np.pi * X['pickup_dayofweek'] / 7).astype(np.float32)
X['day_cos'] = np.cos(2 * np.pi * X['pickup_dayofweek'] / 7).astype(np.float32)

X['is_weekend'] = (X['pickup_dayofweek'] >= 5).astype(np.int8)
X['is_rush_hour'] = ((X['pickup_dayofweek'] < 5) & (X['pickup_hour'].between(7,9) | X['pickup_hour'].between(16,19))).astype(np.int8)

center_lat, center_lon = 37.7879, -122.4074
lat_rad, lon_rad = np.radians(X['pickup_lat']), np.radians(X['pickup_lon'])
c_lat_rad, c_lon_rad = np.radians(center_lat), np.radians(center_lon)
dlat = c_lat_rad - lat_rad
dlon = c_lon_rad - lon_rad
a = np.sin(dlat/2)**2 + np.cos(lat_rad)*np.cos(c_lat_rad)*np.sin(dlon/2)**2
c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
X['dist_to_center'] = (6371 * c).astype(np.float32)

y_bearing = np.sin(dlon) * np.cos(c_lat_rad)
x_bearing = np.cos(lat_rad)*np.sin(c_lat_rad) - np.sin(lat_rad)*np.cos(c_lat_rad)*np.cos(dlon)
X['bearing_to_center'] = ((np.degrees(np.arctan2(y_bearing, x_bearing)) + 360) % 360).astype(np.float32)

PICKUP_GRID = 30
pickup_lat_bins = np.linspace(SF_LAT_MIN, SF_LAT_MAX, PICKUP_GRID+1)
pickup_lon_bins = np.linspace(SF_LON_MIN, SF_LON_MAX, PICKUP_GRID+1)
X['pickup_cell'] = ((np.digitize(trips['pickup_lat'], pickup_lat_bins) - 1) * PICKUP_GRID +
                    (np.digitize(trips['pickup_lon'], pickup_lon_bins) - 1)).astype(np.int16)

y_cont = trips[['dropoff_lat', 'dropoff_lon']].values.astype(np.float32)

Train/Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y_cont, test_size=0.2, random_state=42)

Continuous Spatial Target Encoding

In [8]:
X_train_df = X_train.copy()
X_train_df['target_lat'] = y_train[:, 0]
X_train_df['target_lon'] = y_train[:, 1]

time_stats = X_train_df.groupby(['pickup_cell', 'pickup_hour'])[['target_lat', 'target_lon']].mean().reset_index()
time_stats.rename(columns={'target_lat': 'hist_lat_time', 'target_lon': 'hist_lon_time'}, inplace=True)

static_stats = X_train_df.groupby('pickup_cell')[['target_lat', 'target_lon']].mean().reset_index()
static_stats.rename(columns={'target_lat': 'hist_lat_static', 'target_lon': 'hist_lon_static'}, inplace=True)

global_lat = float(y_train[:, 0].mean())
global_lon = float(y_train[:, 1].mean())

def merge_regression_baselines(df):
    df_out = df.reset_index().merge(time_stats, on=['pickup_cell', 'pickup_hour'], how='left')
    df_out = df_out.merge(static_stats, on='pickup_cell', how='left')
    df_out['hist_lat_time'] = df_out['hist_lat_time'].fillna(df_out['hist_lat_static']).fillna(global_lat).astype(np.float32)
    df_out['hist_lon_time'] = df_out['hist_lon_time'].fillna(df_out['hist_lon_static']).fillna(global_lon).astype(np.float32)
    df_out['hist_lat_static'] = df_out['hist_lat_static'].fillna(global_lat).astype(np.float32)
    df_out['hist_lon_static'] = df_out['hist_lon_static'].fillna(global_lon).astype(np.float32)
    df_out.set_index('index', inplace=True)
    df_out.index.name = None
    cols = ['pickup_lat', 'pickup_lon', 'pickup_hour', 'pickup_dayofweek', 'hour_sin', 'hour_cos',
            'day_sin', 'day_cos', 'is_weekend', 'is_rush_hour', 'dist_to_center', 'bearing_to_center',
            'pickup_cell', 'hist_lat_time', 'hist_lon_time', 'hist_lat_static', 'hist_lon_static']
    return df_out[cols]

X_train = merge_regression_baselines(X_train)
X_test = merge_regression_baselines(X_test)

Train XGBRegressor Model

In [9]:
print("Training Multi-Output XGBoost:")
reg_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    tree_method='hist', device='cuda', random_state=42
)
reg_model.fit(X_train, y_train)
print("done.")

Training Multi-Output XGBoost:
done.


Evaluate on Haversine Metric

In [10]:
def haversine_distance(true_coords, pred_coords):
    R = 6371.0
    lat1, lon1 = np.radians(true_coords[:, 0]), np.radians(true_coords[:, 1])
    lat2, lon2 = np.radians(pred_coords[:, 0]), np.radians(pred_coords[:, 1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

preds = reg_model.predict(X_test)
error_km = np.mean(haversine_distance(y_test, preds))
print(f"Final Spatial Error: {error_km:.2f} km")

Final Spatial Error: 2.04 km


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [19:18:33] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Export models and Meta

In [11]:
export_data = {
    'model': reg_model,
    'time_stats': time_stats,
    'static_stats': static_stats,
    'global_coords': (global_lat, global_lon),
    'bounds': {'lat_min': SF_LAT_MIN, 'lat_max': SF_LAT_MAX, 'lon_min': SF_LON_MIN, 'lon_max': SF_LON_MAX},
    'grid_size': PICKUP_GRID,
    'feature_cols': list(X_train.columns)
}
joblib.dump(export_data, 'sf_taxi_regression_production.pkl')
files.download('sf_taxi_regression_production.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>